In [1]:
import pandas as pd
import cloudscraper
import requests
import json
from datetime import date, timedelta
from io import BytesIO, StringIO
import psycopg2

In [2]:
# Automatizar o download dos dados do investing (cotação do dolar e petroleo)

url_base = "https://api.investing.com/api/financialdata/historical"
id_dolar = 2103
id_petroleo = 8833

scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'firefox',
        'platform': 'linux',
        'mobile': False
    }
)

header_request = {
    'domain-id': 'br',
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:128.0) Gecko/20100101 Firefox/128.0',
    'Accept': '*/*',
    'Accept-Language': 'pt-BR,pt;q=0.8,en-US;q=0.5,en;q=0.3',
    'Referer': 'https://br.investing.com/',
    'Origin': 'https://br.investing.com',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site'
}

scraper.headers.update(header_request)
 
# Intervalo de datas
hoje = date.today()
dias_ate_domingo = (hoje.weekday() + 1) % 7
ultimo_domingo_valido = hoje - timedelta(days=dias_ate_domingo + 7)
ultimo_sabado_valido = ultimo_domingo_valido + timedelta(days=6)
data_final = hoje - timedelta(days= 365)


In [ ]:
# Automatizar o download dos dados da ANP (preço dos combustíveis)
inicio_str = ultimo_domingo_valido
fim_str = ultimo_sabado_valido

df_combustiveis = pd.DataFrame()

while inicio_str >= data_final:
    try:
        url = ''
        if inicio_str <= date(2025, 10, 12):
            url = f"https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/{inicio_str.year}/revendas_lpc_{inicio_str}_{fim_str}.xlsx"
        else:
            if(inicio_str == date(2025, 10, 19)):
                url = f'https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/{inicio_str.year}/revendas_lpc.xlsx'
            else:    
                diferenca_dias = inicio_str - date(2025, 10, 19)
                id_url = int(diferenca_dias.days/7)
                url = f'https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/{inicio_str.year}/revendas_lpc-{id_url}.xlsx'

        response = requests.get(url)
        response.raise_for_status()

        xlsx_bytes = BytesIO(response.content)
        df = pd.read_excel(xlsx_bytes)

        df.columns = df.iloc[8]
        df = df.iloc[9:]
        df = df.reset_index(drop=True)

        df_combustiveis = pd.concat([df_combustiveis, df], ignore_index=True)

    except Exception as e:
        print(f"Erro ao carregar de {inicio_str} a {fim_str}:\n {e}")

    finally:
        inicio_str -= timedelta(days=7)
        fim_str -= timedelta(days=7)

if not df_combustiveis.empty:
    colunas_uteis = {
        'CNPJ': 'cnpj',
        'RAZÃO': 'razão_social',
        'FANTASIA': 'nome_fantasia',
        'ESTADO': 'estado',
        'MUNICÍPIO': 'municipio',
        'PRODUTO': 'produto',
        'PREÇO DE REVENDA': 'preco',
        'DATA DA COLETA': 'data_coleta'
    }

    colunas_finais_combustiveis = []
    for col_orig, col_nova in colunas_uteis.items():
        if(col_orig in df_combustiveis.columns):
            df_combustiveis[col_nova] = df_combustiveis[col_orig]
            colunas_finais_combustiveis.append(col_nova)

    df_combustiveis = df_combustiveis[colunas_finais_combustiveis]

    df_combustiveis['cnpj'] = df_combustiveis['cnpj'].astype(str).str.replace(r'[^0-9]', '', regex=True).zfill(14)

    df_combustiveis = df_combustiveis.astype({
        'razão_social': 'string',
        'nome_fantasia': 'string',
        'estado': 'string',
        'municipio': 'string',
        'produto': 'string',
    })

    df_combustiveis['preco'] = pd.to_numeric(df_combustiveis['preco'], errors='coerce', downcast='float')

    df_combustiveis['data_coleta'] = pd.to_datetime(df_combustiveis['data_coleta'])

    #df_combustiveis.to_csv(f'arquivo_final.csv', index=False, header=True, sep=';')

    #print(df_combustiveis)

else:
    print('\n\nNenhym dado foi carregado')

Erro ao carregar de 2025-11-16 a 2025-11-22:
 404 Client Error: Not Found for url: https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/2025/revendas_lpc-4.xlsx
Erro ao carregar de 2024-12-29 a 2025-01-04:
 404 Client Error: Not Found for url: https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/2024/revendas_lpc_2024-12-29_2025-01-04.xlsx
Erro ao carregar de 2024-12-22 a 2024-12-28:
 404 Client Error: Not Found for url: https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/2024/revendas_lpc_2024-12-22_2024-12-28.xlsx


In [4]:
try:
    # montar url para baixar cotação petroleo
    url = f"{url_base}/{id_petroleo}"

    data_inicio = ultimo_domingo_valido

    params_request = {
        'start-date': data_final,
        'end-date': data_inicio,
        'time-frame': 'Weekly',
        'add-missing-rows': 'false'
    }

    response = scraper.get(url, params=params_request, timeout=30)
    response.raise_for_status()

    dados = response.json()

    df_petroleo = pd.DataFrame(dados['data'])

    if not df_petroleo.empty:
        colunas_uteis = {
            'last_close': 'preco',
            'rowDate': 'data_cotacao',
            'volumeRaw': 'volume_transacoes'
        }

        colunas_finais_petroleo = []
        for col_orig, col_nova in colunas_uteis.items():
            if(col_orig in df_petroleo.columns):
                df_petroleo[col_nova] = df_petroleo[col_orig]
                colunas_finais_petroleo.append(col_nova)

        df_petroleo = df_petroleo[colunas_finais_petroleo]

        df_petroleo['preco'] = pd.to_numeric(df_petroleo['preco'].str.replace(',', '.'), errors='coerce').round(2).astype(float)

        df_petroleo['data_cotacao'] = pd.to_datetime(df_petroleo['data_cotacao'], format = '%d.%m.%Y')

        df_petroleo['volume_transacoes'] = df_petroleo['volume_transacoes'].astype(int)

        print(df_petroleo)

    else:
        print('Nenhum dado baixado')

except requests.exceptions.RequestException as e:
    print(f"Erro na requisição: {e}")

    preco data_cotacao  volume_transacoes
0   62.56   2025-11-16            1693462
1   64.39   2025-11-09            1771454
2   63.63   2025-11-02            1658600
3   65.07   2025-10-26             716309
4   65.94   2025-10-19            2125757
5   61.29   2025-10-12            1896365
6   62.28   2025-10-05            1142663
7   64.53   2025-09-28            1339594
8   70.13   2025-09-21            1046041
9   66.68   2025-09-14            1277324
10  66.99   2025-09-07            1486361
11  65.11   2025-08-31            1321396
12  68.12   2025-08-24             563686
13  67.73   2025-08-17            1288098
14  65.85   2025-08-10            1288292
15  66.59   2025-08-03            1734115
16  68.76   2025-07-27            2012393
17  68.44   2025-07-20            1324211
18  69.28   2025-07-13            1775344
19  69.16   2025-07-06            1154113
20  67.22   2025-06-29            1117124
21  67.77   2025-06-22            1521423
22  77.01   2025-06-15            

In [ ]:
try:
    # montar url para baixar cotação dolar
    url = f"{url_base}/{id_dolar}"

    data_inicio = ultimo_domingo_valido
    
    params_request = {
        'start-date': data_final,
        'end-date': data_inicio,
        'time-frame': 'Weekly',
        'add-missing-rows': 'false'
    }

    response = scraper.get(url, params=params_request, timeout=30)
    response.raise_for_status()

    dados = response.json()

    df_dolar = pd.DataFrame(dados['data'])

    if not df_dolar.empty:
        colunas_uteis = {
            'last_close': 'preco',
            'rowDate': 'data_cotacao',
        }
        colunas_finais_dolar = []
        for col_orig, col_nova in colunas_uteis.items():
            if(col_orig in df_dolar.columns):
                df_dolar[col_nova] = df_dolar[col_orig]
                colunas_finais_dolar.append(col_nova)

        df_dolar = df_dolar[colunas_finais_dolar]

        df_dolar['preco'] = pd.to_numeric(df_dolar['preco'].str.replace(',', '.'), errors='coerce').round(4).astype(float)

        df_dolar['data_cotacao'] = pd.to_datetime(df_dolar['data_cotacao'], format = '%d.%m.%Y')

        print(df_dolar)
    
    else:
        print("Nenhum dado carregado")

except requests.exceptions.RequestException as e:
    print(f"Erro na requisição: {e}")

     preco data_cotacao
0   5.4033   2025-11-16
1   5.2965   2025-11-09
2   5.3324   2025-11-02
3   5.3744   2025-10-26
4   5.3896   2025-10-19
5   5.4085   2025-10-12
6   5.5220   2025-10-05
7   5.3375   2025-09-28
8   5.3431   2025-09-21
9   5.3238   2025-09-14
10  5.3549   2025-09-07
11  5.4134   2025-08-31
12  5.4304   2025-08-24
13  5.4227   2025-08-17
14  5.4009   2025-08-10
15  5.4340   2025-08-03
16  5.5420   2025-07-27
17  5.5633   2025-07-20
18  5.5790   2025-07-13
19  5.5589   2025-07-06
20  5.4212   2025-06-29
21  5.4856   2025-06-22
22  5.5132   2025-06-15
23  5.5439   2025-06-08
24  5.5594   2025-06-01
25  5.7227   2025-05-25
26  5.6468   2025-05-18
27  5.6633   2025-05-11
28  5.6556   2025-05-04
29  5.6579   2025-04-27
30  5.6841   2025-04-20
31  5.8073   2025-04-13
32  5.8656   2025-04-06
33  5.8425   2025-03-30
34  5.7604   2025-03-23
35  5.7291   2025-03-16
36  5.7385   2025-03-09
37  5.7874   2025-03-02
38  5.8852   2025-02-23
39  5.7320   2025-02-16
40  5.7042   202

In [ ]:
conn = psycopg2.connect(
    dbname="predicao_combustiveis",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

cur = conn.cursor()

query = """
CREATE TABLE IF NOT EXISTS estado (
    id SERIAL PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    sigla CHAR(2) NOT NULL UNIQUE
);
"""
cur.execute(query)

query = """
CREATE TABLE IF NOT EXISTS cidade (
    id SERIAL PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    estado_id INT NOT NULL, 
    CONSTRAINT fk_estado FOREIGN KEY (estado_id) REFERENCES estado(id)
);
"""
cur.execute(query)

query = """
CREATE TABLE IF NOT EXISTS posto (
    id SERIAL PRIMARY KEY,
    cnpj VARCHAR(14) NOT NULL UNIQUE,
    razao_social VARCHAR(100) NOT NULL,
    nome_fantasia VARCHAR(100) NULL,
    cidade_id INT NOT NULL, 
    CONSTRAINT fk_cidade FOREIGN KEY (cidade_id) REFERENCES cidade(id)
);
"""
cur.execute(query)

query = """
CREATE TABLE IF NOT EXISTS combustiveis (
    id SERIAL PRIMARY KEY,
    combustivel VARCHAR(30) NOT NULL UNIQUE
);
"""
cur.execute(query)

query = """
CREATE TABLE IF NOT EXISTS cotacao_combustiveis (
    id SERIAL PRIMARY KEY,
    posto_id INT NOT NULL,
    combustivel_id INT NOT NULL,
    preco NUMERIC(4, 2) NOT NULL,
    data_coleta DATE NOT NULL,
    CONSTRAINT fk_posto FOREIGN KEY (posto_id) REFERENCES posto(id),
    CONSTRAINT fk_combustivel FOREIGN KEY (combustivel_id) REFERENCES combustiveis(id),
    UNIQUE(posto_id, combustivel_id, preco, data_coleta)
);  
"""
cur.execute(query)

query = """
CREATE TABLE IF NOT EXISTS cotacao_dolar (
    id SERIAL PRIMARY KEY,
    preco NUMERIC(5, 4) NOT NULL,
    data_cotacao DATE NOT NULL
);
"""
cur.execute(query)

query = """
CREATE TABLE IF NOT EXISTS cotacao_petroleo (
    id SERIAL PRIMARY KEY,
    preco NUMERIC(4, 2) NOT NULL,
    data_cotacao DATE NOT NULL,
    volume_transacoes INT NOT NULL
);
"""
cur.execute(query)

query = """
INSERT INTO estado (nome, sigla) VALUES
    ('ACRE', 'AC'),
    ('ALAGOAS', 'AL'),
    ('AMAPA', 'AP'),
    ('AMAZONAS', 'AM'),
    ('BAHIA', 'BA'),
    ('CEARA', 'CE'),
    ('DISTRITO FEDERAL', 'DF'),
    ('ESPIRITO SANTO', 'ES'),
    ('GOIAS', 'GO'),
    ('MARANHAO', 'MA'),
    ('MATO GROSSO', 'MT'),
    ('MATO GROSSO DO SUL', 'MS'),
    ('MINAS GERAIS', 'MG'),
    ('PARA', 'PA'),
    ('PARAIBA', 'PB'),
    ('PARANA', 'PR'),
    ('PERNAMBUCO', 'PE'),
    ('PIAUI', 'PI'),
    ('RIO DE JANEIRO', 'RJ'),
    ('RIO GRANDE DO NORTE', 'RN'),
    ('RIO GRANDE DO SUL', 'RS'),
    ('RONDONIA', 'RO'),
    ('RORAIMA', 'RR'),
    ('SANTA CATARINA', 'SC'),
    ('SAO PAULO', 'SP'),
    ('SERGIPE', 'SE'),
    ('TOCANTINS', 'TO');
"""
cur.execute(query)

query = """
INSERT INTO combustiveis (combustivel) VALUES
    ('DIESEL S10'),
    ('GASOLINA COMUM'),
    ('DIESEL S500'),
    ('GNV'),
    ('ETANOL'),
    ('GASOLINA ADITIVADA');
"""
cur.execute(query)

conn.commit()

cur.close()
conn.close()

In [ ]:
df_combustiveis_csv = StringIO()
df_combustiveis[df_combustiveis['estado'] == 'MATO GROSSO DO SUL'].to_csv(df_combustiveis_csv, index=False, header=False, sep=';')
df_combustiveis_csv.seek(0)

df_dolar_csv = StringIO()
df_dolar.to_csv(df_dolar_csv, index=False, header=False, sep=';')
df_dolar_csv.seek(0)

df_petroleo_csv = StringIO()
df_petroleo.to_csv(df_petroleo_csv, index=False, header=False, sep=';')
df_petroleo_csv.seek(0)

conn = psycopg2.connect(
    dbname="predicao_combustiveis",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

cur = conn.cursor()

query = """
CREATE TEMP TABLE IF NOT EXISTS dados_anp (
    cnpj VARCHAR(14) NULL,
    razao_social VARCHAR(100) NULL,
    nome_fantasia VARCHAR(100) NULL,
    cidade VARCHAR(100) NULL,
    estado VARCHAR(50) NULL,
    produto VARCHAR(50) NULL,
    preco NUMERIC(6, 2) NULL,
    data_coleta DATE NULL
);
"""
cur.execute(query)

cur.copy_from(
    df_combustiveis_csv,
    'dados_anp',
    sep=';',
    columns=('cnpj', 'razao_social', 'nome_fantasia', 'estado', 'cidade', 'produto', 'preco', 'data_coleta')
)

cur.copy_from(
    df_dolar_csv,
    'cotacao_dolar',
    sep = ';',
    columns = ('preco', 'data_cotacao')
)

cur.copy_from(
    df_petroleo_csv,
    'cotacao_petroleo',
    sep = ';',
    columns = ('preco', 'data_cotacao', 'volume_transacoes')
)

query = """
INSERT INTO cidade (nome, estado_id) 
    SELECT DISTINCT cidade, estado.id 
        FROM dados_anp 
        JOIN estado ON estado.nome = dados_anp.estado 
    WHERE NOT EXISTS (SELECT 1 FROM cidade WHERE cidade.nome = dados_anp.cidade AND cidade.estado_id = estado.id) 
    ORDER BY 1;
"""
cur.execute(query)

query = """
INSERT INTO posto (cnpj, razao_social, nome_fantasia, cidade_id) 
    SELECT  
        cnpj, 
        MAX(razao_social), 
        MAX(nome_fantasia), 
        cidade.id 
    FROM dados_anp 
    JOIN cidade ON cidade.nome = dados_anp.cidade 
    JOIN estado ON estado.nome = dados_anp.estado 
                    AND estado.id = cidade.estado_id
    WHERE 
        produto <> 'GLP' 
        AND NOT EXISTS (SELECT 1 FROM posto WHERE posto.cnpj = dados_anp.cnpj)
    GROUP BY cnpj, cidade.id;
"""
cur.execute(query)

query = """
INSERT INTO cotacao_combustiveis (posto_id, combustivel_id, preco, data_coleta)
    SELECT posto.id, combustiveis.id, dados_anp.preco, dados_anp.data_coleta 
        FROM dados_anp 
        JOIN posto ON posto.cnpj = dados_anp.cnpj 
        JOIN combustiveis ON combustiveis.combustivel = dados_anp.produto 
    WHERE 
        dados_anp.estado = 'MATO GROSSO DO SUL'
        AND NOT EXISTS (
            SELECT 1 
            FROM cotacao_combustiveis 
            WHERE
                cotacao_combustiveis.posto_id = posto.id
                AND cotacao_combustiveis.combustivel_id = combustiveis.id
                AND cotacao_combustiveis.preco = dados_anp.preco
                AND cotacao_combustiveis.data_coleta = dados_anp.data_coleta
            )  
    ORDER BY 4;
"""
cur.execute(query)

conn.commit()

cur.close()
conn.close()